# SuperBook Local Image → Video proof

This is an isolated proof of real AI motion. It does **not** modify the SuperBook Flutter app. Upload one SuperBook scene image, describe the motion, and generate an MP4 with LTX-Video.

The notebook uses the official LTX-Video inference extra. The previous version used the wrong optional-dependency name, which left `av` unavailable in Colab.

In [ ]:
!git clone -q --depth 1 https://github.com/Lightricks/LTX-Video.git
%cd LTX-Video
!python -m pip install -q -e '.[inference]'
!python -c "import av, imageio, torchvision; import ltx_video; print('LTX dependencies OK')"

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print('Image:', IMAGE_PATH)

In [ ]:
MOTION_PROMPT = '''A storybook character moves naturally through the scene. The character takes two slow steps toward the table, turns toward the other character, raises one hand while speaking, with subtle breathing and natural clothing movement. Preserve the exact character identity, face, clothing, room, lighting and composition. No new characters, no scene change, no camera cuts, no text.'''
print(MOTION_PROMPT)

In [ ]:
!python inference.py \
  --prompt "$MOTION_PROMPT" \
  --conditioning_media_paths "$IMAGE_PATH" \
  --conditioning_start_frames 0 \
  --height 512 \
  --width 768 \
  --num_frames 97 \
  --seed 42 \
  --pipeline_config configs/ltxv-2b-0.9.6-distilled.yaml

In [ ]:
import glob, os
from IPython.display import Video, display
videos = sorted(glob.glob('/content/LTX-Video/**/*.mp4', recursive=True), key=os.path.getmtime, reverse=True)
print(videos[:3])
if not videos:
    raise FileNotFoundError('No MP4 was generated.')
display(Video(videos[0], embed=True))

## Decision gate

If the generated clip shows **real character/body movement** while preserving the scene, this engine is a candidate for the SuperBook video backend. If it only produces camera drift, warping, or negligible motion, we discard it and test the next engine.